# 01 - Análise exploratória: score de crédito (QuantumFinance)

Objetivo: entender o dataset antes de definir o pipeline de treino (`src/treinamento.py`).
Este notebook é **exploratório**: nenhuma regra de negócio vive aqui. Tudo o que for
decidido (limpeza, features, janela temporal) é implementado em `src/credit_score/processamento.py`,
que é o código usado em treino, inferência e API.

In [1]:
import sys
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
import pandas as pd
from credit_score.processamento import FEATURES, MESES, limpar_dados
treino = pd.read_csv(RAIZ / "data/raw/train.csv", low_memory=False)
teste = pd.read_csv(RAIZ / "data/raw/test.csv", low_memory=False)
print("treino:", treino.shape, "| teste:", teste.shape)

treino: (100000, 28) | teste: (50000, 27)


## 1. Estrutura temporal
O treino cobre janeiro a agosto e o teste setembro a dezembro, para os mesmos 12.500 clientes. Por isso a validação é **temporal** (último mês), e não aleatória: uma divisão aleatória misturaria meses do mesmo cliente e inflaria as métricas.

In [2]:
print(treino["Month"].value_counts().reindex(MESES).dropna().astype(int).to_dict())
print(teste["Month"].value_counts().reindex(MESES).dropna().astype(int).to_dict())
print("clientes em comum:", len(set(treino.Customer_ID) & set(teste.Customer_ID)))

{'January': 12500, 'February': 12500, 'March': 12500, 'April': 12500, 'May': 12500, 'June': 12500, 'July': 12500, 'August': 12500}
{'September': 12500, 'October': 12500, 'November': 12500, 'December': 12500}
clientes em comum: 12500


## 2. Variável alvo
Classes desbalanceadas: a métrica principal escolhida é o **F1 macro** e os modelos usam `class_weight='balanced'`.

In [3]:
treino["Credit_Score"].value_counts(normalize=True).round(3)

Credit_Score
Standard    0.532
Poor        0.290
Good        0.178
Name: proportion, dtype: float64

## 3. Qualidade dos dados
O dataset tem sujeira proposital: números como texto com `_`, sentinelas (`_______`, `!@9#%8`), valores impossíveis (idade -500, 1.798 contas).

In [4]:
for col in ["Age", "Num_Bank_Accounts", "Interest_Rate", "Credit_Mix", "Payment_Behaviour"]:
    print(col, "->", treino[col].astype(str).value_counts().head(4).to_dict())

Age -> {'38': 2833, '28': 2829, '31': 2806, '26': 2792}
Num_Bank_Accounts -> {'6': 13001, '7': 12823, '8': 12765, '4': 12186}
Interest_Rate -> {'8': 5012, '5': 4979, '6': 4721, '12': 4540}
Credit_Mix -> {'Standard': 36479, 'Good': 24337, '_': 20195, 'Bad': 18989}
Payment_Behaviour -> {'Low_spent_Small_value_payments': 25513, 'High_spent_Medium_value_payments': 17540, 'Low_spent_Medium_value_payments': 13861, 'High_spent_Large_value_payments': 13721}


In [5]:
limpo = limpar_dados(treino)
resumo = pd.DataFrame({"nulos_apos_limpeza_%": (limpo.isna().mean() * 100).round(1)})
resumo[resumo["nulos_apos_limpeza_%"] > 0]

,nulos_apos_limpeza_%
Age,2.8
Annual_Income,1.0
Monthly_Inhand_Salary,15.0
Num_Bank_Accounts,1.3
Num_Credit_Card,2.3
Interest_Rate,2.0
Num_of_Loan,4.3
Num_of_Delayed_Payment,8.4
Changed_Credit_Limit,2.1
Num_Credit_Inquiries,3.6


## 4. Relação das variáveis com o alvo
As variáveis comportamentais (atrasos, mix de crédito, pagamento mínimo) separam bem as classes.

In [6]:
base = limpo.assign(Credit_Score=treino["Credit_Score"])
base.groupby("Credit_Score")[["Delay_from_due_date", "Num_of_Delayed_Payment", "Outstanding_Debt",
                              "Interest_Rate", "Credit_History_Age"]].median().round(1)

,Delay_from_due_date,Num_of_Delayed_Payment,Outstanding_Debt,Interest_Rate,Credit_History_Age
Credit_Score,,,,,
Good,10.0,8.0,736.5,7.0,288.0
Poor,27.0,17.0,1945.4,21.0,161.0
Standard,18.0,14.0,1001.6,13.0,227.0


In [7]:
pd.crosstab(base["Credit_Mix"], base["Credit_Score"], normalize="index").round(2)

Credit_Score,Good,Poor,Standard
Credit_Mix,,,
Bad,0.02,0.60,0.38
Desconhecido,0.18,0.29,0.53
Good,0.49,0.16,0.35
Standard,0.06,0.22,0.73


## 5. Decisões levadas para o pipeline
| Decisão | Motivo |
|---|---|
| Treinar só com os **4 meses mais recentes** (`janela_meses_recentes`) | requisito de negócio: modelo baseado nas transações mais recentes |
| Validação **out-of-time** no último mês | evita vazamento entre meses do mesmo cliente |
| Remover `ID`, `Customer_ID`, `Name`, `SSN` | identificadores / dados pessoais (LGPD) sem poder preditivo |
| Remover `Type_of_Loan`, `Occupation`, `Month` | modelo mais simples; alta cardinalidade ou pouco sinal |
| Valores fora de faixa → nulo → mediana | limpeza robusta e igual em treino e produção |
| F1 macro como métrica principal + recall mínimo de `Poor` | desbalanceamento e controle de risco |